# Processamento, Limpeza e Análise dos Dados

## 1. Processamento dos dados

In [0]:
%python
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd

# Paths dos arquivos no Workspace
BASE_PATH = "/Workspace/Users/matheusmartinez2018@gmail.com/coupon-optimization-ml/data/raw"

OFFERS_PATH = f"{BASE_PATH}/offers.json" # ids das ofertas e metadados de cada uma delas 
PROFILE_PATH = f"{BASE_PATH}/profile.json" # atributos de cerca de 17k cliente
TRANSACTIONS_PATH = f"{BASE_PATH}/transactions.json" # cerca de 300k eventos

print("paths:")
print(f"offers: {OFFERS_PATH}")
print(f"profile: {PROFILE_PATH}")
print(f"transactions: {TRANSACTIONS_PATH}")

In [0]:
# Leitura dos dados
df_offers       = spark.read.option("multiline", "true").json(OFFERS_PATH)
df_profile      = spark.read.option("multiline", "true").json(PROFILE_PATH)
df_transactions = spark.read.option("multiline", "true").json(TRANSACTIONS_PATH)

# Dimensões de cada dataset
print(" - OFFERS - ")
print(f"Linhas: {df_offers.count()} | Colunas: {len(df_offers.columns)}")
df_offers.printSchema()

print(" - PROFILE - ")
print(f"Linhas: {df_profile.count()} | Colunas: {len(df_profile.columns)}")
df_profile.printSchema()

print(" - TRANSACTIONS - ")
print(f"Linhas: {df_transactions.count()} | Colunas: {len(df_transactions.columns)}")
df_transactions.printSchema()

## 2. Análise Descritiva

In [0]:

print(" OFFERS - todas as linhas ")
df_offers.show(truncate=False)

print(" PROFILE - amostra ")
df_profile.show(5, truncate=False)

print(" TRANSACTIONS - amostra por tipo de evento ")
df_transactions.groupBy("event").count().orderBy("count", ascending=False).show()

print(" TRANSACTIONS - amostra de cada tipo de evento ")
for event_type in ["offer received", "offer viewed", "offer completed", "transaction"]:
    print(f"\n--- {event_type} ---")
    df_transactions.filter(F.col("event") == event_type).show(2, truncate=False)

In [0]:
# Distribuições, nulos e contagens iniciais
print(" PROFILE — Dist. das Idades ")
df_profile.groupBy("age").count().orderBy("count", ascending=False).show(10)

print(" PROFILE — Nulos por Coluna ")
df_profile.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_profile.columns
]).show()

print(" PROFILE — Dist. de Gênero ")
df_profile.groupBy("gender").count().orderBy("count", ascending=False).show()

print(" TRANSACTIONS — Clientes Únicos ")
print(f"Clientes únicos nas transações: {df_transactions.select('account_id').distinct().count()}")
print(f"Clientes únicos no profile:     {df_profile.select('id').distinct().count()}")

print(" TRANSACTIONS — Range Temporal ")
df_transactions.select(
    F.min("time_since_test_start").alias("dia_inicio"),
    F.max("time_since_test_start").alias("dia_fim")
).show()

print(" OFFERS — Tipos de Oferta ")
df_offers.groupBy("offer_type").count().show()

In [0]:
# Coleta os dados necessários para os gráficos via PySpark
# e converte para listas — matplotlib não lê DataFrame Spark

# Profile
ages = [r['age'] for r in df_profile.select('age').collect()]
cc_limits = [r['credit_card_limit'] for r in df_profile
             .filter(F.col('credit_card_limit').isNotNull())
             .select('credit_card_limit').collect()]
gender_dist = {r['gender']: r['count'] for r in df_profile
               .groupBy('gender').count().collect()}

# Transações
event_dist = {r['event']: r['count'] for r in df_transactions
              .groupBy('event').count().collect()}
amounts = [r['amount'] for r in df_transactions
           .filter(F.col('event') == 'transaction')
           .select(F.col('value.amount').alias('amount')).collect()]
time_dist = [r['time_since_test_start'] for r in df_transactions
             .select('time_since_test_start').collect()]

# ── Grid ──────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 14))
fig.suptitle('Análise Exploratória — iFood Coupon Dataset', 
             fontsize=16, fontweight='bold', y=0.98)

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# 1. Distribuição de idade (com o spike em 118)
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist([a for a in ages if a != 118], bins=30, color='#2A6CF0', edgecolor='white', alpha=0.85)
ax1.axvline(118, color='red', linestyle='--', linewidth=1.5, label='placeholder (118)')
ax1.set_title('Distribuição de Idade')
ax1.set_xlabel('Idade')
ax1.set_ylabel('Frequência')
ax1.legend(fontsize=8)

# 2. Boxplot de idade (para ver outlier claramente)
ax2 = fig.add_subplot(gs[0, 1])
ax2.boxplot(ages, vert=True, patch_artist=True,
            boxprops=dict(facecolor='#2A6CF0', alpha=0.6))
ax2.set_title('Boxplot — Idade')
ax2.set_ylabel('Idade')

# 3. Distribuição de gênero
ax3 = fig.add_subplot(gs[0, 2])
labels = [str(k) for k in gender_dist.keys()]
values = list(gender_dist.values())
colors = ['#2A6CF0', '#1D3557', '#90ADC6', '#E63946']
ax3.bar(labels, values, color=colors, edgecolor='white')
ax3.set_title('Distribuição de Gênero')
ax3.set_ylabel('Clientes')
for i, v in enumerate(values):
    ax3.text(i, v + 50, str(v), ha='center', fontsize=8)

# 4. Distribuição de limite do cartão
ax4 = fig.add_subplot(gs[1, 0])
ax4.hist(cc_limits, bins=40, color='#1D3557', edgecolor='white', alpha=0.85)
ax4.set_title('Limite do Cartão de Crédito')
ax4.set_xlabel('Limite (R$)')
ax4.set_ylabel('Frequência')

# 5. Boxplot de limite do cartão (outliers)
ax5 = fig.add_subplot(gs[1, 1])
ax5.boxplot(cc_limits, vert=True, patch_artist=True,
            boxprops=dict(facecolor='#1D3557', alpha=0.6))
ax5.set_title('Boxplot — Limite do Cartão')
ax5.set_ylabel('Limite (R$)')

# 6. Tipos de evento
ax6 = fig.add_subplot(gs[1, 2])
ev_labels = list(event_dist.keys())
ev_values = list(event_dist.values())
bars = ax6.barh(ev_labels, ev_values, color='#2A6CF0', edgecolor='white', alpha=0.85)
ax6.set_title('Volume por Tipo de Evento')
ax6.set_xlabel('Quantidade')
for bar, v in zip(bars, ev_values):
    ax6.text(v + 500, bar.get_y() + bar.get_height()/2, 
             f'{v:,}', va='center', fontsize=8)

# 7. Distribuição do ticket (amount)
ax7 = fig.add_subplot(gs[2, 0])
ax7.hist([a for a in amounts if a < 100], bins=50, 
         color='#2A6CF0', edgecolor='white', alpha=0.85)
ax7.set_title('Ticket por Transação (até R$100)')
ax7.set_xlabel('Valor (R$)')
ax7.set_ylabel('Frequência')

# 8. Boxplot de ticket
ax8 = fig.add_subplot(gs[2, 1])
ax8.boxplot(amounts, vert=True, patch_artist=True,
            boxprops=dict(facecolor='#2A6CF0', alpha=0.6))
ax8.set_title('Boxplot — Ticket')
ax8.set_ylabel('Valor (R$)')

# 9. Distribuição temporal dos eventos
ax9 = fig.add_subplot(gs[2, 2])
ax9.hist(time_dist, bins=30, color='#1D3557', edgecolor='white', alpha=0.85)
ax9.set_title('Distribuição Temporal dos Eventos')
ax9.set_xlabel('Dia do experimento')
ax9.set_ylabel('Frequência')

#plt.savefig('/Workspace/Users/matheusmartinez2018@gmail.com/coupon-optimization-ml/data/processed/eda_grid.png',
 #           dpi=150, bbox_inches='tight')
plt.show()
print("Grid salvo em data/processed/eda_grid.png")

In [0]:
# Investigação: age=118, distribuição de tickets, picos temporais e limite do cartão.

# 1. Confirmando que age=118 tem sempre nulos em gender e credit_card
print("1. age=118 são exatamente os mesmos nulos?")

total_118     = df_profile.filter(F.col('age') == 118).count()
sem_gender_188 = df_profile.filter((F.col('age') == 118) & F.col('gender').isNotNull()).count()
sem_cc_118     = df_profile.filter((F.col('age') == 118) & F.col('credit_card_limit').isNotNull()).count()
nulos_fora_118_gender = df_profile.filter((F.col('age') != 118) & F.col('gender').isNull()).count()
nulos_fora_118_cc     = df_profile.filter((F.col('age') != 118) & F.col('credit_card_limit').isNull()).count()

print(f"Perfis com age=118:                          {total_118}")
print(f"Desses, com gender preenchido:               {sem_gender_188}")
print(f"Desses, com credit_card preenchido:          {sem_cc_118}")
print(f"Nulos em gender FORA de age=118:             {nulos_fora_118_gender}")
print(f"Nulos em credit_card FORA de age=118:        {nulos_fora_118_cc}")
# VERIFICADO! 

# 2. Tickets muito altos
print("2. Distribuição dos tickets altos")

df_amounts = df_transactions \
    .filter(F.col('event') == 'transaction') \
    .select(F.col('value.amount').alias('amount'))

df_amounts.select(
    F.count('amount').alias('total_transacoes'),
    F.round(F.mean('amount'), 2).alias('media'),
    F.round(F.stddev('amount'), 2).alias('desvio_padrao'),
    F.round(F.percentile_approx('amount', 0.25), 2).alias('p25'),
    F.round(F.percentile_approx('amount', 0.50), 2).alias('p50_mediana'),
    F.round(F.percentile_approx('amount', 0.75), 2).alias('p75'),
    F.round(F.percentile_approx('amount', 0.90), 2).alias('p90'),
    F.round(F.percentile_approx('amount', 0.95), 2).alias('p95'),
    F.round(F.percentile_approx('amount', 0.99), 2).alias('p99'),
    F.round(F.max('amount'), 2).alias('maximo')
).show()

# Quantas transações acima de cada limite (definidos arbitrariamente)?
print("Transações acima de R$100:")
print(df_amounts.filter(F.col('amount') > 100).count())
print("Transações acima de R$200:")
print(df_amounts.filter(F.col('amount') > 200).count())
print("Transações acima de R$500:")
print(df_amounts.filter(F.col('amount') > 500).count())

# 3. Picos temporais 
print("\n 3. Volume de eventos por dia")

df_transactions \
    .withColumn('dia', F.floor('time_since_test_start').cast('int')) \
    .groupBy('dia', 'event') \
    .count() \
    .orderBy('dia', 'event') \
    .show(60)

#  4. Investigar limite do cartão 
print("\n 4. Estatísticas do limite do cartão")

df_profile.filter(F.col('credit_card_limit').isNotNull()) \
    .select(
        F.round(F.mean('credit_card_limit'), 2).alias('media'),
        F.round(F.stddev('credit_card_limit'), 2).alias('desvio_padrao'),
        F.round(F.percentile_approx('credit_card_limit', 0.25), 2).alias('p25'),
        F.round(F.percentile_approx('credit_card_limit', 0.50), 2).alias('p50'),
        F.round(F.percentile_approx('credit_card_limit', 0.75), 2).alias('p75'),
        F.round(F.percentile_approx('credit_card_limit', 0.95), 2).alias('p95'),
        F.round(F.min('credit_card_limit'), 2).alias('minimo'),
        F.round(F.max('credit_card_limit'), 2).alias('maximo')
    ).show()

### Achados

**Ofertas**

10 ofertas no catálogo: 4 BOGO, 4 discount e 2 informacionais. As informacionais não possuem evento de conclusão, o que impede medir conversão a partir dos dados. Serão excluídas do modelo por essa razão.

Os canais disponíveis são web, email, mobile e social, com cobertura variando **por oferta**.

**Perfil dos Clientes**

- 2.175 clientes (12,8%) têm cadastro incompleto: age=118, gender=NULL e credit_card_limit=NULL sempre juntos. Não existe nulo em gender ou credit_card fora desse grupo, nem age=118 com dados preenchidos.
- Distribuição de gênero entre os perfis completos: 49,9% M, 36,1% F, 1,2% O.
- Limite do cartão vai de R$30k a R$120k, mediana R$64k, sem assimetria relevante.

**Transações**

- 4 tipos de eventos numa janela de 30 dias. 
- Os 17.000 clientes do profile aparecem integralmente nas transações. 
- 99% dos tickets estão abaixo de R$40, com mediana de R$8,89. Existem 478 tickets acima de R$100 e 185 acima de R$500. São atípicos, mas plausíveis para uma plataforma com categorias como mercado e farmácia (considerando que os dados seriam do Ifood).

Também foi encontrada uma inconsistência na estrutura do banco. O offer_id aparece sob duas chaves distintas dentro
do campo `value`. Nos eventos received e viewed, a chave é `"offer id"` com espaço. No evento completed, é `"offer_id"` com underscore. Precisa ser normalizado antes de qualquer join.

O padrão temporal revela o delineamento do experimento: `offer received` aparece em massa nos dias 0, 7, 14 e 17. As ofertas foram disparadas em ondas semanais, não de forma contínua. Essa informação foi crucial para a contrução dos grupos de treino, validação e teste no modelo. 

---

### Premissas

1. age=118 é cadastro incompleto, não outlier de idade. Substituído por nulo com flag `incomplete_profile`;
2. Gênero O é categoria válida, mantido;
3. `"offer id"` e `"offer_id"` representam o mesmo campo;
4. Tickets acima de R$100 são atípicos mas aceitáveis pro modelo de negócio, mantidos com flag `high_ticket`;
5. Ofertas informacionais serão excluídas do modelo por ausência de evento de conclusão (que será nossa variável resposta);
6. Limites de cartão mantidos sem normalização por ausência de referência externa.

## 3. Limpeza e Normalização

### Arquitetura de Dados (Medallion)

Os dados passam por duas camadas de transformação antes de chegar ao modelo, simulando a arquitetura Medallion (isso garante uma separação clara entre processamento e modelagem). 

- **Silver** — tabela `silver_events`. Limpeza e normalização dos três datasets brutos em uma única tabela de eventos, com `offer_id` unificado, campo `value` explodido em colunas, flags de qualidade (`incomplete_profile`, `high_ticket`) e enriquecimento com metadados de ofertas e perfil do cliente.

Granularidade: uma linha por evento.

- **Gold** — tabela `gold_episodios`. Agregação da Silver na granularidade de episódio, ou seja, um envio específico de uma oferta para um cliente em um momento no tempo. Cada episódio carrega o tempo de recebimento, visualização
(quando existe) e conclusão (quando existe), respeitando a janela de validade da oferta. É essa a granularidade correta para medir resposta e alimentar o modelo de recomendação de cupons.

Granularidade: uma linha por (account_id, offer_id, received_time).

Ambas as camadas são construídas neste notebook. O notebook de modelagem carrega a Gold, constrói as features e treina o modelo.

In [0]:
# Construção da Silver: limpeza, normalização e enriquecimento dos eventos brutos

# Normaliza offer_id (aparece com duas grafias diferentes no campo value)
df_transactions_clean = df_transactions \
    .withColumn(
        "offer_id_clean",
        F.coalesce(F.col("value.`offer id`"), F.col("value.offer_id"))
    ) \
    .withColumn("amount", F.col("value.amount")) \
    .withColumn("reward", F.col("value.reward")) \
    .withColumn("day", F.floor("time_since_test_start").cast("int")) \
    .withColumn(
        "high_ticket",
        F.when(
            (F.col("event") == "transaction") & (F.col("value.amount") > 100), 1
        ).otherwise(0)
    ) \
    .drop("value")

# Normaliza profile: converte data, cria flag de cadastro incompleto e substitui age=118 por nulo
df_profile_clean = df_profile \
    .withColumn(
        "registered_on",
        F.to_date(F.col("registered_on").cast("string"), "yyyyMMdd")
    ) \
    .withColumn(
        "incomplete_profile",
        F.when(F.col("age") == 118, 1).otherwise(0)
    ) \
    .withColumn(
        "age",
        F.when(F.col("age") == 118, None).otherwise(F.col("age"))
    )

# Prepara ofertas com alias limpo para os joins
df_offers_prep = df_offers \
    .withColumn("n_channels", F.size("channels")) \
    .select(
        F.col("id").alias("offer_id_ref"),
        F.col("offer_type"),
        F.col("discount_value"),
        F.col("min_value"),
        F.col("duration"),
        F.col("n_channels")
    )

# Prepara profile com alias limpo para os joins
df_profile_prep = df_profile_clean \
    .select(
        F.col("id").alias("account_id_ref"),
        F.col("age"),
        F.col("gender"),
        F.col("credit_card_limit"),
        F.col("registered_on"),
        F.col("incomplete_profile")
    )

# Enriquece eventos com metadados da oferta e do cliente
df_silver = df_transactions_clean \
    .join(
        df_offers_prep,
        df_transactions_clean["offer_id_clean"] == df_offers_prep["offer_id_ref"],
        how="left"
    ) \
    .drop("offer_id_ref") \
    .join(
        df_profile_prep,
        df_transactions_clean["account_id"] == df_profile_prep["account_id_ref"],
        how="left"
    ) \
    .drop("account_id_ref") \
    .select(
        "account_id", "event", "day", "time_since_test_start",
        "offer_id_clean", "amount", "reward", "high_ticket",
        "offer_type", "discount_value", "min_value", "duration", "n_channels",
        "age", "gender", "credit_card_limit", "registered_on", "incomplete_profile"
    ) \
    .orderBy("account_id", "time_since_test_start")

print("Linhas na Silver:", df_silver.count(), "| esperado 306534")
df_silver.printSchema()

In [0]:
from pyspark.sql.window import Window

# Reconstroi df_offers_prep para uso nos joins de episodios
df_offers_prep = df_offers \
    .withColumn("n_channels", F.size("channels")) \
    .select(
        F.col("id").alias("offer_id_ref"),
        F.col("offer_type"),
        F.col("discount_value"),
        F.col("min_value"),
        F.col("duration"),
        F.col("n_channels")
    )

# Usa time_since_test_start (valor decimal) para ordenacao e associacao
# A coluna day = floor(time_since_test_start) perde precisao intradiaria
# e nao deve ser usada para ordenacao temporal de eventos

window_recebimento = Window.partitionBy("account_id", "offer_id_clean") \
    .orderBy("time_since_test_start")

df_recebimentos = df_silver \
    .filter(F.col("event") == "offer received") \
    .withColumn("episodio", F.row_number().over(window_recebimento)) \
    .select(
        "account_id",
        F.col("offer_id_clean").alias("offer_id"),
        F.col("time_since_test_start").alias("received_time"),
        F.col("day").alias("received_day"),
        "episodio"
    )

# Isola viewed e completed brutos da Silver com tempo decimal
df_viewed_raw = df_silver \
    .filter(F.col("event") == "offer viewed") \
    .select(
        "account_id",
        F.col("offer_id_clean").alias("offer_id"),
        F.col("time_since_test_start").alias("viewed_time"),
        F.col("day").alias("viewed_day")
    )

df_completed_raw = df_silver \
    .filter(F.col("event") == "offer completed") \
    .select(
        "account_id",
        F.col("offer_id_clean").alias("offer_id"),
        F.col("time_since_test_start").alias("completed_time"),
        F.col("day").alias("completed_day"),
        F.col("reward")
    )

# Associa cada viewed ao episodio correto usando tempo decimal.
# Valida que viewed_time esta dentro da janela de validade da oferta.
# Preserva apenas a primeira visualizacao por episodio
df_viewed_assoc = df_viewed_raw \
    .join(df_recebimentos, on=["account_id", "offer_id"], how="inner") \
    .join(
        df_offers_prep.select("offer_id_ref", "duration"),
        df_viewed_raw["offer_id"] == df_offers_prep["offer_id_ref"],
        how="left"
    ) \
    .filter(F.col("viewed_time") >= F.col("received_time")) \
    .filter(F.col("viewed_time") <= F.col("received_time") + F.col("duration")) \
    .withColumn(
        "rank_proximidade",
        F.row_number().over(
            Window.partitionBy("account_id", "offer_id", "viewed_time")
                  .orderBy(F.desc("received_time"))
        )
    ) \
    .filter(F.col("rank_proximidade") == 1) \
    .withColumn(
        "rank_episodio",
        F.row_number().over(
            Window.partitionBy("account_id", "offer_id", "episodio")
                  .orderBy("viewed_time")
        )
    ) \
    .filter(F.col("rank_episodio") == 1) \
    .select("account_id", "offer_id", "episodio", "viewed_time", "viewed_day")

# Associa completed ao episodio correto usando tempo decimal.
# Valida que completed_time esta dentro da janela de validade da oferta.
# Agrega reward por momento de conclusao antes de resolver o episodio,
# preservando o valor total quando o cliente completou mais de uma vez
df_completed_por_tempo = df_completed_raw \
    .join(
        df_offers_prep.select("offer_id_ref", "duration"),
        df_completed_raw["offer_id"] == df_offers_prep["offer_id_ref"],
        how="left"
    ) \
    .groupBy("account_id", "offer_id", "completed_time", "completed_day") \
    .agg(
        F.sum("reward").alias("reward"),
        F.count("*").alias("n_eventos_no_tempo"),
        F.first("duration").alias("duration")
    )

df_completed_assoc_tempo = df_completed_por_tempo \
    .join(df_recebimentos, on=["account_id", "offer_id"], how="inner") \
    .filter(F.col("completed_time") >= F.col("received_time")) \
    .filter(F.col("completed_time") <= F.col("received_time") + F.col("duration")) \
    .withColumn(
        "rank_proximidade",
        F.row_number().over(
            Window.partitionBy("account_id", "offer_id", "completed_time")
                  .orderBy(F.desc("received_time"))
        )
    ) \
    .filter(F.col("rank_proximidade") == 1)

df_completed_assoc = df_completed_assoc_tempo \
    .groupBy("account_id", "offer_id", "episodio") \
    .agg(
        F.min("completed_time").alias("completed_time"),
        F.min("completed_day").alias("completed_day"),
        F.sum("reward").alias("reward"),
        F.sum("n_eventos_no_tempo").alias("n_conclusoes_no_episodio")
    )

print("Recebimentos:", df_recebimentos.count(), "| esperado 76277")
print("Viewed assoc:", df_viewed_assoc.count())
print("Completed assoc:", df_completed_assoc.count())
print("Reward total:", df_completed_assoc.agg(F.sum("reward")).collect()[0][0])


### Construção da tabela de episódios

A granularidade original dos eventos, uma linha por acontecimento, não é suficiente para medir resposta a uma oferta com precisão. O mesmo cliente pode receber a mesma oferta mais de uma vez no período, e um cruzamento simples por cliente e oferta acabaria misturando envios diferentes.

Cada envio é tratado como um episódio independente. A associação entre visualização, conclusão e recebimento é feita pelo tempo decimal (`time_since_test_start`), garantindo precisão intradiária. O `day` inteiro é mantido só para agregações visuais.

Cada episódio carrega o campo `valid_until = received_time + duration`, que delimita a janela de validade daquele envio específico. Só visualizações e conclusões dentro dessa janela são associadas ao episódio. Eventos fora do prazo são descartados.

### Achados

- **1.158 visualizações foram descartadas por acontecerem após o vencimento da oferta.**

Nenhuma delas resultou em conclusão.

- **478 episódios tiveram mais de uma conclusão dentro da mesma janela de validade.**

Para o modelo, foi mantida uma linha por episódio (a pergunta é **binária**: esse envio gerou resposta?). O reward de todas as conclusões dentro do mesmo episódio é somado, preservando o valor total pago em desconto. Soma validada: R$164.676.

- **17,6% das conclusões aconteceram sem visualização registrada dentro da janela.**

Isso pode indicar  compras **não influenciadas pelas ofertas**, mas também não comprova que a compra teria ocorrido sem ela. O dataset **não permite** essa afirmação causal. Esse grupo entra no modelo como variável observada, sem interpretação causal.

In [0]:
# Validação da Silver: preservação de eventos, nulos esperados e flags aplicadas

# 1. Linhas preservadas por tipo de evento
print(" EVENTOS PRESERVADOS ")
df_silver.groupBy("event").count().orderBy("count", ascending=False).show()

# 2. Nulos por coluna
print(" NULOS POR COLUNA ")
df_silver.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_silver.columns
]).show()

# 3. Verificar se offer_id_clean nulo so aparece em transacoes
print(" offer_id_clean NULO POR EVENTO ")
df_silver \
    .filter(F.col("offer_id_clean").isNull()) \
    .groupBy("event") \
    .count() \
    .show()

# 4. Verificar se offer_type nulo so aparece em transacoes
print(" offer_type NULO POR EVENTO ")
df_silver \
    .filter(F.col("offer_type").isNull()) \
    .groupBy("event") \
    .count() \
    .show()

# 5. Verificar age nulo apos substituicao do 118
print(" IDADE NULA (esperado: 2175) ")
print(df_silver.filter(F.col("age").isNull()).select("account_id").distinct().count())

# 6. Verificar high_ticket
print(" HIGH TICKET ")
print(f"Transacoes com high_ticket=1: {df_silver.filter(F.col('high_ticket') == 1).count()}")
print(f"Esperado: 478")

# 7. Verificar registered_on convertido corretamente
print(" REGISTERED_ON ")
df_silver.select("registered_on").distinct().orderBy("registered_on").show(5)

In [0]:
# Monta a Gold (gold_episodios): une recebimentos, viewed, completed,
# metadados da oferta e perfil do cliente numa linha por episódio.
# valid_until = received_time + duration delimita a janela de validade individual.


# Reconstroi df_profile_prep para o join com episodios
df_profile_prep = df_profile_clean \
    .select(
        F.col("id").alias("account_id_ref"),
        F.col("age"),
        F.col("gender"),
        F.col("credit_card_limit"),
        F.col("registered_on"),
        F.col("incomplete_profile")
    )

# Monta a tabela final de episodios incluindo perfil do cliente.
# valid_until marca o prazo de validade de cada episodio especifico,
# usado para filtrar eventos fora da janela e para definir censura temporal
df_episodios = df_recebimentos \
    .join(df_viewed_assoc, on=["account_id", "offer_id", "episodio"], how="left") \
    .join(df_completed_assoc, on=["account_id", "offer_id", "episodio"], how="left") \
    .join(
        df_offers_prep,
        df_recebimentos["offer_id"] == df_offers_prep["offer_id_ref"],
        how="left"
    ) \
    .drop("offer_id_ref") \
    .join(
        df_profile_prep,
        df_recebimentos["account_id"] == df_profile_prep["account_id_ref"],
        how="left"
    ) \
    .drop("account_id_ref") \
    .withColumn("valid_until", F.col("received_time") + F.col("duration"))

print("Total na tabela final:", df_episodios.count(), "| esperado 76277")
print("Colunas:", df_episodios.columns)
print("Soma de reward:", df_episodios.agg(F.sum("reward")).collect()[0][0], "| esperado 164676.0")
df_episodios.show(3, truncate=False)

In [0]:
# Salva a Gold como tabela Delta para consumo no notebook de modelagem
df_episodios.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_episodios")

print("gold_episodios salva com sucesso")
print("Total:", spark.table("gold_episodios").count())

## 4. Análise de Negócio

In [0]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# Análise do funil de ofertas: conversão por oferta, tipo, duração, valor mínimo, desconto e canais

# Coleta dos dados para as analises
# Funil por oferta individual
df_funil_oferta = df_episodios \
    .filter(F.col("offer_type") != "informational") \
    .groupBy("offer_id", "offer_type", "discount_value", "min_value", "duration", "n_channels") \
    .agg(
        F.count("*").alias("recebidas"),
        F.count(F.when(F.col("viewed_day").isNotNull(), 1)).alias("visualizadas"),
        F.count(F.when(F.col("completed_day").isNotNull(), 1)).alias("completadas")
    ) \
    .withColumn("taxa_visualizacao", F.round(F.col("visualizadas") / F.col("recebidas") * 100, 1)) \
    .withColumn("taxa_conversao", F.round(F.col("completadas") / F.col("recebidas") * 100, 1)) \
    .withColumn("taxa_conclusao_pos_view", F.round(F.col("completadas") / F.col("visualizadas") * 100, 1)) \
    .orderBy("taxa_conversao", ascending=False)

funil = df_funil_oferta.toPandas()

# "Conclusão com vs sem visualização\n(episódios com validade corrigida)"
df_organico = df_episodios \
    .filter(
        (F.col("offer_type") != "informational") &
        (F.col("completed_day").isNotNull())
    ) \
    .withColumn(
        "organico",
        F.when(F.col("viewed_day").isNull(), 1).otherwise(0)
    )

organico = df_organico.groupBy("organico").count().toPandas()
total_comp = organico["count"].sum()
n_organico = organico[organico["organico"] == 1]["count"].values[0] if 1 in organico["organico"].values else 0
n_influenciado = total_comp - n_organico

# Conversao por tipo de oferta
tipo_conv = df_funil_oferta \
    .groupBy("offer_type") \
    .agg(
        F.sum("recebidas").alias("recebidas"),
        F.sum("completadas").alias("completadas")
    ) \
    .withColumn("taxa", F.round(F.col("completadas") / F.col("recebidas") * 100, 1)) \
    .toPandas()

# ── FIGURA 1 ────────────────────────────────────────────────
fig = plt.figure(figsize=(20, 16))
fig.suptitle("Análise do Funil de Ofertas", fontsize=16, fontweight="bold", y=0.98)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.5, wspace=0.4)

cores = ["#2A6CF0", "#1D3557", "#457B9D", "#A8DADC", "#E63946",
         "#F4A261", "#2EC4B6", "#FF9F1C"]

# 1. Taxa de conversao por oferta (barras horizontais)
ax1 = fig.add_subplot(gs[0, :2])
bars = ax1.barh(
    [f"{row['offer_type'].upper()} | desc={row['discount_value']} | min={row['min_value']} | dur={int(row['duration'])}d"
     for _, row in funil.iterrows()],
    funil["taxa_conversao"],
    color=cores[:len(funil)], edgecolor="white", alpha=0.9
)
for bar, v in zip(bars, funil["taxa_conversao"]):
    ax1.text(v + 0.3, bar.get_y() + bar.get_height()/2, f"{v}%", va="center", fontsize=9)
ax1.set_title("Taxa de Conversão por Oferta (%)", fontweight="bold")
ax1.set_xlabel("Conversão (%)")
ax1.axvline(funil["taxa_conversao"].mean(), color="red", linestyle="--", linewidth=1, label=f"Média: {funil['taxa_conversao'].mean():.1f}%")
ax1.legend(fontsize=8)

# 2. Funil agregado: recebidas vs visualizadas vs completadas
ax2 = fig.add_subplot(gs[0, 2])
etapas = ["Recebidas", "Visualizadas", "Completadas"]
valores = [funil["recebidas"].sum(), funil["visualizadas"].sum(), funil["completadas"].sum()]
ax2.barh(etapas, valores, color=["#1D3557", "#2A6CF0", "#A8DADC"], edgecolor="white")
for i, v in enumerate(valores):
    ax2.text(v + 200, i, f"{v:,}", va="center", fontsize=9)
ax2.set_title("Funil Geral de Ofertas\n(exceto informacional)", fontweight="bold")

# 3. "Conclusão com vs sem visualização"
ax3 = fig.add_subplot(gs[1, 0])
labels = ["Com visualização", "Sem visualização)"]
sizes = [n_influenciado, n_organico]
colors_pie = ["#2A6CF0", "#E63946"]
wedges, texts, autotexts = ax3.pie(
    sizes, labels=labels, colors=colors_pie,
    autopct="%1.1f%%", startangle=90,
    textprops={"fontsize": 9}
)
ax3.set_title("Conclusão com vs sem visualização", fontweight="bold")

# 4. Conversao por tipo de oferta
ax4 = fig.add_subplot(gs[1, 1])
ax4.bar(tipo_conv["offer_type"], tipo_conv["taxa"],
        color=["#2A6CF0", "#1D3557"], edgecolor="white", alpha=0.9)
for i, row in tipo_conv.iterrows():
    ax4.text(i, row["taxa"] + 0.3, f"{row['taxa']}%", ha="center", fontsize=9)
ax4.set_title("Taxa de Conversão por Tipo", fontweight="bold")
ax4.set_ylabel("Conversão (%)")

# 5. Conversao por duracao da oferta
ax5 = fig.add_subplot(gs[1, 2])
dur_conv = funil.groupby("duration").agg(
    recebidas=("recebidas", "sum"),
    completadas=("completadas", "sum")
).reset_index()
dur_conv["taxa"] = (dur_conv["completadas"] / dur_conv["recebidas"] * 100).round(1)
ax5.bar(dur_conv["duration"].astype(str) + "d", dur_conv["taxa"],
        color="#457B9D", edgecolor="white", alpha=0.9)
for i, row in dur_conv.iterrows():
    ax5.text(i, row["taxa"] + 0.3, f"{row['taxa']}%", ha="center", fontsize=9)
ax5.set_title("Taxa de Conversão por Duração", fontweight="bold")
ax5.set_ylabel("Conversão (%)")

# 6. Conversao por valor minimo da oferta
ax6 = fig.add_subplot(gs[2, 0])
minval_conv = funil.groupby("min_value").agg(
    recebidas=("recebidas", "sum"),
    completadas=("completadas", "sum")
).reset_index()
minval_conv["taxa"] = (minval_conv["completadas"] / minval_conv["recebidas"] * 100).round(1)
ax6.bar(minval_conv["min_value"].astype(str), minval_conv["taxa"],
        color="#2EC4B6", edgecolor="white", alpha=0.9)
for i, row in minval_conv.iterrows():
    ax6.text(i, row["taxa"] + 0.3, f"{row['taxa']}%", ha="center", fontsize=9)
ax6.set_title("Taxa de Conversão por Valor Mínimo (R$)", fontweight="bold")
ax6.set_ylabel("Conversão (%)")
ax6.set_xlabel("Valor mínimo (R$)")

# 7. Conversao por valor de desconto
ax7 = fig.add_subplot(gs[2, 1])
desc_conv = funil.groupby("discount_value").agg(
    recebidas=("recebidas", "sum"),
    completadas=("completadas", "sum")
).reset_index()
desc_conv["taxa"] = (desc_conv["completadas"] / desc_conv["recebidas"] * 100).round(1)
ax7.bar(desc_conv["discount_value"].astype(str), desc_conv["taxa"],
        color="#F4A261", edgecolor="white", alpha=0.9)
for i, row in desc_conv.iterrows():
    ax7.text(i, row["taxa"] + 0.3, f"{row['taxa']}%", ha="center", fontsize=9)
ax7.set_title("Taxa de Conversão por Valor de Desconto (R$)", fontweight="bold")
ax7.set_ylabel("Conversão (%)")
ax7.set_xlabel("Desconto (R$)")

# 8. Conversao por numero de canais
ax8 = fig.add_subplot(gs[2, 2])
canal_conv = funil.groupby("n_channels").agg(
    recebidas=("recebidas", "sum"),
    completadas=("completadas", "sum")
).reset_index()
canal_conv["taxa"] = (canal_conv["completadas"] / canal_conv["recebidas"] * 100).round(1)
ax8.bar(canal_conv["n_channels"].astype(str) + " canais", canal_conv["taxa"],
        color="#FF9F1C", edgecolor="white", alpha=0.9)
for i, row in canal_conv.iterrows():
    ax8.text(i, row["taxa"] + 0.3, f"{row['taxa']}%", ha="center", fontsize=9)
ax8.set_title("Taxa de Conversão por Nº de Canais", fontweight="bold")
ax8.set_ylabel("Conversão (%)")

plt.savefig("/Workspace/Users/matheusmartinez2018@gmail.com/coupon-optimization-ml/data/processed/eda_funil.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Figura 1 salva.")

### Funil de Ofertas

O funil promocional (exceto informacionais) registrou 61 mil episódios recebidos, 76% de taxa de visualização e 54% de taxa de conclusão entre quem visualizou.

As duas ofertas de maior conversão são descontos de R$2 e R$3 com prazo de 7 a 10 dias e valor mínimo de R$7 a R$10, ambas acima de 65%. BOGOs com desconto de R$10 ficam abaixo de 48%. O padrão é consistente: **acessibilidade da oferta pesa mais do que o tamanho do incentivo.**

Achados que se destacaram:  
- Duração importa: ofertas de 10 dias convertem 56% contra 50% das de 5 dias. 
- O valor mínimo tem ponto ótimo em R$7 (66,7%), com queda a partir de R$10. 
- Mais canais geram mais conversão: 4 canais chegam a 58,9% contra 43,2% com 2 canais. Como os canais são
fixos por oferta, não é possível separar o efeito do canal do efeito da oferta com esses dados.

- 17% das conclusões aconteceram sem visualização registrada dentro da janela de
validade. Esse número foi recalculado com a granularidade correta de episódios,
que associa cada conclusão ao envio específico ao qual pertence. Esses casos
indicam possível subsídio de compras não influenciadas pela oferta, mas o dado
observacional não permite afirmar causalidade.

In [0]:
# Análise bivariada: relação entre perfil do cliente e propensão a converter
# Features calculadas com transações ANTERIORES ao recebimento da oferta (sem leakage)

df_transacoes = df_silver \
    .filter(F.col("event") == "transaction") \
    .select("account_id", F.col("time_since_test_start").alias("transaction_time"), "amount")

df_ticket = df_episodios \
    .join(df_transacoes, on="account_id", how="left") \
    .filter(F.col("transaction_time") < F.col("received_time")) \
    .groupBy("account_id", "offer_id", "episodio", "received_time", "received_day",
             "completed_day", "viewed_day", "offer_type",
             "discount_value", "min_value", "duration",
             "age", "gender", "credit_card_limit", "incomplete_profile") \
    .agg(
        F.count("amount").alias("n_transacoes_anteriores"),
        F.avg("amount").alias("ticket_medio_anterior"),
        F.sum("amount").alias("gasto_total_anterior"),
        F.max("transaction_time").alias("ultima_transacao_time")
    ) \
    .withColumn(
        "ticket_ratio",
        F.round(F.col("ticket_medio_anterior") / F.col("min_value"), 2)
    ) \
    .withColumn(
        "converteu",
        F.when(F.col("completed_day").isNotNull(), 1).otherwise(0)
    ) \
    .withColumn(
        "recencia",
        F.when(
            F.col("ultima_transacao_time").isNotNull(),
            F.col("received_time") - F.col("ultima_transacao_time")
        ).otherwise(None)
    )

df_ticket_filtrado = df_ticket \
    .filter(F.col("offer_type") != "informational") \
    .filter(F.col("n_transacoes_anteriores") > 0) \
    .filter(F.col("min_value") > 0)

print("Episodios com historico de transacao:", df_ticket_filtrado.count())

ticket_pd = df_ticket_filtrado.select(
    "ticket_ratio", "converteu", "ticket_medio_anterior",
    "n_transacoes_anteriores", "gasto_total_anterior",
    "recencia", "age", "gender", "credit_card_limit",
    "offer_type", "discount_value", "min_value"
).toPandas()

fig = plt.figure(figsize=(20, 16))
fig.suptitle("Análise Bivariada — Perfil do Cliente e Propensão à Oferta",
             fontsize=16, fontweight="bold", y=0.98)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.5, wspace=0.4)

# 1. Ticket ratio por conversao
ax1 = fig.add_subplot(gs[0, 0])
grupos = [
    ticket_pd[ticket_pd["converteu"] == 0]["ticket_ratio"].dropna().clip(0, 10),
    ticket_pd[ticket_pd["converteu"] == 1]["ticket_ratio"].dropna().clip(0, 10)
]
bp = ax1.boxplot(grupos, patch_artist=True, tick_labels=["Não converteu", "Converteu"])
bp["boxes"][0].set_facecolor("#E63946")
bp["boxes"][1].set_facecolor("#2A6CF0")
ax1.set_title("Ticket Ratio por Conversão\n(ticket_medio / min_value, cap=10)", fontweight="bold")
ax1.set_ylabel("Ticket Ratio")
ax1.axhline(1, color="gray", linestyle="--", linewidth=1, label="ratio=1 (ticket = min_value)")
ax1.legend(fontsize=8)

# 2. Ticket ratio por faixa
ax2 = fig.add_subplot(gs[0, 1])
bins = [0, 0.5, 1, 1.5, 2, 3, 5, 100]
labels_bins = ["<0.5", "0.5-1", "1-1.5", "1.5-2", "2-3", "3-5", ">5"]
ticket_pd["ticket_ratio_faixa"] = pd.cut(ticket_pd["ticket_ratio"], bins=bins, labels=labels_bins)
conv_ratio = ticket_pd.groupby("ticket_ratio_faixa", observed=True)["converteu"].mean() * 100
ax2.bar(conv_ratio.index, conv_ratio.values, color="#2A6CF0", edgecolor="white", alpha=0.9)
for i, v in enumerate(conv_ratio.values):
    ax2.text(i, v + 0.5, f"{v:.1f}%", ha="center", fontsize=8)
ax2.set_title("Conversão por Faixa de Ticket Ratio", fontweight="bold")
ax2.set_ylabel("Taxa de Conversão (%)")
ax2.set_xlabel("ticket_medio / min_value")

# 3. Frequencia historica
ax3 = fig.add_subplot(gs[0, 2])
bins_freq = [0, 2, 5, 10, 20, 200]
labels_freq = ["1-2", "3-5", "6-10", "11-20", ">20"]
ticket_pd["freq_faixa"] = pd.cut(ticket_pd["n_transacoes_anteriores"],
                                   bins=bins_freq, labels=labels_freq)
conv_freq = ticket_pd.groupby("freq_faixa", observed=True)["converteu"].mean() * 100
ax3.bar(conv_freq.index, conv_freq.values, color="#1D3557", edgecolor="white", alpha=0.9)
for i, v in enumerate(conv_freq.values):
    ax3.text(i, v + 0.5, f"{v:.1f}%", ha="center", fontsize=8)
ax3.set_title("Conversão por Frequência Histórica", fontweight="bold")
ax3.set_ylabel("Taxa de Conversão (%)")
ax3.set_xlabel("Transações antes do envio")

# 4. Recencia
ax4 = fig.add_subplot(gs[1, 0])
bins_rec = [0, 2, 5, 10, 20, 100]
labels_rec = ["0-2d", "3-5d", "6-10d", "11-20d", ">20d"]
ticket_pd["recencia_faixa"] = pd.cut(ticket_pd["recencia"], bins=bins_rec, labels=labels_rec)
conv_rec = ticket_pd.groupby("recencia_faixa", observed=True)["converteu"].mean() * 100
ax4.bar(conv_rec.index, conv_rec.values, color="#457B9D", edgecolor="white", alpha=0.9)
for i, v in enumerate(conv_rec.values):
    ax4.text(i, v + 0.5, f"{v:.1f}%", ha="center", fontsize=8)
ax4.set_title("Conversão por Recência\n(dias desde última transação)", fontweight="bold")
ax4.set_ylabel("Taxa de Conversão (%)")
ax4.set_xlabel("Dias desde última compra")

# 5. Genero
ax5 = fig.add_subplot(gs[1, 1])
conv_genero = ticket_pd.groupby("gender")["converteu"].mean() * 100
conv_genero = conv_genero.dropna()
cores_g = {"M": "#1D3557", "F": "#2A6CF0", "O": "#A8DADC"}
cores_lista = [cores_g.get(g, "#ccc") for g in conv_genero.index]
ax5.bar(conv_genero.index, conv_genero.values, color=cores_lista, edgecolor="white", alpha=0.9)
for i, v in enumerate(conv_genero.values):
    ax5.text(i, v + 0.3, f"{v:.1f}%", ha="center", fontsize=9)
ax5.set_title("Conversão por Gênero", fontweight="bold")
ax5.set_ylabel("Taxa de Conversão (%)")

# 6. Faixa etaria
ax6 = fig.add_subplot(gs[1, 2])
ticket_pd_age = ticket_pd[ticket_pd["age"].notna() & (ticket_pd["age"] < 100)]
bins_age = [18, 30, 40, 50, 60, 70, 100]
labels_age = ["18-30", "31-40", "41-50", "51-60", "61-70", "70+"]
ticket_pd_age = ticket_pd_age.copy()
ticket_pd_age["age_faixa"] = pd.cut(ticket_pd_age["age"], bins=bins_age, labels=labels_age)
conv_age = ticket_pd_age.groupby("age_faixa", observed=True)["converteu"].mean() * 100
ax6.bar(conv_age.index, conv_age.values, color="#2EC4B6", edgecolor="white", alpha=0.9)
for i, v in enumerate(conv_age.values):
    ax6.text(i, v + 0.3, f"{v:.1f}%", ha="center", fontsize=8)
ax6.set_title("Conversão por Faixa Etária", fontweight="bold")
ax6.set_ylabel("Taxa de Conversão (%)")

# 7. Ticket ratio por tipo de oferta
ax7 = fig.add_subplot(gs[2, 0])
grupos_tipo = [
    ticket_pd[ticket_pd["offer_type"] == "bogo"]["ticket_ratio"].dropna().clip(0, 10),
    ticket_pd[ticket_pd["offer_type"] == "discount"]["ticket_ratio"].dropna().clip(0, 10)
]
bp2 = ax7.boxplot(grupos_tipo, patch_artist=True, tick_labels=["BOGO", "Discount"])
bp2["boxes"][0].set_facecolor("#2A6CF0")
bp2["boxes"][1].set_facecolor("#1D3557")
ax7.set_title("Ticket Ratio por Tipo de Oferta", fontweight="bold")
ax7.set_ylabel("Ticket Ratio (cap=10)")
ax7.axhline(1, color="red", linestyle="--", linewidth=1, alpha=0.7)

# 8. Gasto total anterior
ax8 = fig.add_subplot(gs[2, 1])
nao_conv = ticket_pd[ticket_pd["converteu"] == 0]["gasto_total_anterior"].clip(0, 500)
sim_conv = ticket_pd[ticket_pd["converteu"] == 1]["gasto_total_anterior"].clip(0, 500)
ax8.hist(nao_conv, bins=40, alpha=0.6, color="#E63946", label="Não converteu", density=True)
ax8.hist(sim_conv, bins=40, alpha=0.6, color="#2A6CF0", label="Converteu", density=True)
ax8.set_title("Gasto Total Anterior por Conversão\n(até R$500)", fontweight="bold")
ax8.set_xlabel("Gasto total anterior (R$)")
ax8.set_ylabel("Densidade")
ax8.legend(fontsize=8)

# 9. Limite de credito
ax9 = fig.add_subplot(gs[2, 2])
ticket_pd_cc = ticket_pd[ticket_pd["credit_card_limit"].notna()]
bins_cc = [0, 45000, 60000, 75000, 90000, 120001]
labels_cc = ["<45k", "45-60k", "60-75k", "75-90k", ">90k"]
ticket_pd_cc = ticket_pd_cc.copy()
ticket_pd_cc["cc_faixa"] = pd.cut(ticket_pd_cc["credit_card_limit"],
                                    bins=bins_cc, labels=labels_cc)
conv_cc = ticket_pd_cc.groupby("cc_faixa", observed=True)["converteu"].mean() * 100
ax9.bar(conv_cc.index, conv_cc.values, color="#FF9F1C", edgecolor="white", alpha=0.9)
for i, v in enumerate(conv_cc.values):
    ax9.text(i, v + 0.3, f"{v:.1f}%", ha="center", fontsize=8)
ax9.set_title("Conversão por Limite de Crédito", fontweight="bold")
ax9.set_ylabel("Taxa de Conversão (%)")
ax9.set_xlabel("Limite do cartão (R$)")

plt.savefig("/Workspace/Users/matheusmartinez2018@gmail.com/coupon-optimization-ml/data/processed/eda_bivariada.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Figura 2 salva.")

### Perfil do Cliente e Propensão à Oferta

O ticket ratio (_ticket médio / valor mínimo_) foi a variável mais discriminante.
Clientes com ratio abaixo de 0,5 convertem 23% — o gasto habitual deles não alcança
a barreira da oferta. A partir de ratio 1 a conversão sobe para 80% e se estabiliza.
Essa variável entra diretamente no modelo como feature de compatibilidade cliente-oferta.

Frequência histórica tem ponto ótimo em 6 a 10 transações (66%). Acima de 20 transações
a conversão cai para 52%, sugerindo que clientes muito frequentes já compram sem precisar
de incentivo. Clientes com poucas transações (1 a 2) ficam em 50%.

Recência decai linearmente: 60% para quem comprou nos últimos 2 dias contra 48% para
quem está há mais de 20 dias sem comprar. Clientes recentes são mais responsivos.

Gênero F (74%) e O (72%) superam M (56%) com margem expressiva. Conversão cresce com
a idade, chegando a 69% acima dos 70 anos. Limite de crédito tem relação quase linear:
81% acima de R$90k contra 46% abaixo de R$45k.

Todas essas variáveis foram calculadas com dados anteriores ao recebimento da oferta
para evitar leakage no modelo.

In [0]:
# Analise de tempo de resposta por oferta
# Tempo entre recebimento e visualizacao, e entre recebimento e conclusao

df_tempo = df_episodios \
    .filter(F.col("offer_type") != "informational") \
    .withColumn(
        "dias_ate_view",
        F.when(
            F.col("viewed_time").isNotNull(),
            F.col("viewed_time") - F.col("received_time")
        ).otherwise(None)
    ) \
    .withColumn(
        "dias_ate_complete",
        F.when(
            F.col("completed_time").isNotNull(),
            F.col("completed_time") - F.col("received_time")
        ).otherwise(None)
    ) \
    .withColumn(
        "concluiu_nas_ultimas_24h",
        F.when(
            F.col("completed_time").isNotNull(),
            F.when(
                F.col("completed_time") >= F.col("valid_until") - 1,
                1
            ).otherwise(0)
        ).otherwise(None)
    )

# Estatisticas de tempo por oferta
df_tempo_oferta = df_tempo \
    .groupBy("offer_id", "offer_type", "discount_value", "min_value", "duration") \
    .agg(
        F.round(F.avg("dias_ate_view"), 1).alias("avg_dias_ate_view"),
        F.round(F.avg("dias_ate_complete"), 1).alias("avg_dias_ate_complete"),
        F.round(F.avg("concluiu_nas_ultimas_24h") * 100, 1).alias("pct_conclusao_ultimo_dia"),
        F.count(F.when(F.col("completed_day").isNotNull(), 1)).alias("n_completadas")
    ) \
    .orderBy("avg_dias_ate_complete")

tempo_oferta_pd = df_tempo_oferta.toPandas()

# Distribucao de dias ate view e complete para plot
dias_view_pd = df_tempo \
    .filter(F.col("dias_ate_view").isNotNull()) \
    .select("offer_type", "dias_ate_view", "duration") \
    .toPandas()

dias_complete_pd = df_tempo \
    .filter(F.col("dias_ate_complete").isNotNull()) \
    .select("offer_id", "offer_type", "discount_value",
            "min_value", "duration", "dias_ate_complete") \
    .toPandas()

# ── FIGURA 3 ────────────────────────────────────────────────
fig = plt.figure(figsize=(20, 14))
fig.suptitle("Análise de Tempo de Resposta às Ofertas",
             fontsize=16, fontweight="bold", y=0.98)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.4)

# 1. Tempo medio ate visualizacao por oferta
ax1 = fig.add_subplot(gs[0, 0])
labels_oferta = [
    f"{row['offer_type'].upper()}\ndesc={row['discount_value']} min={row['min_value']} dur={int(row['duration'])}d"
    for _, row in tempo_oferta_pd.iterrows()
]
ax1.barh(labels_oferta, tempo_oferta_pd["avg_dias_ate_view"],
         color="#2A6CF0", edgecolor="white", alpha=0.9)
for i, v in enumerate(tempo_oferta_pd["avg_dias_ate_view"]):
    ax1.text(v + 0.05, i, f"{v}d", va="center", fontsize=8)
ax1.set_title("Tempo Médio até Visualização\n(dias após recebimento)", fontweight="bold")
ax1.set_xlabel("Dias")

# 2. Tempo medio ate conclusao por oferta
ax2 = fig.add_subplot(gs[0, 1])
ax2.barh(labels_oferta, tempo_oferta_pd["avg_dias_ate_complete"],
         color="#1D3557", edgecolor="white", alpha=0.9)
for i, v in enumerate(tempo_oferta_pd["avg_dias_ate_complete"]):
    ax2.text(v + 0.05, i, f"{v}d", va="center", fontsize=8)
ax2.set_title("Tempo Médio até Conclusão\n(dias após recebimento)", fontweight="bold")
ax2.set_xlabel("Dias")

# 3. Percentual que conclui no ultimo dia de validade
ax3 = fig.add_subplot(gs[0, 2])
colors_urgencia = ["#E63946" if v > 20 else "#2A6CF0"
                   for v in tempo_oferta_pd["pct_conclusao_ultimo_dia"]]
ax3.barh(labels_oferta, tempo_oferta_pd["pct_conclusao_ultimo_dia"],
         color=colors_urgencia, edgecolor="white", alpha=0.9)
for i, v in enumerate(tempo_oferta_pd["pct_conclusao_ultimo_dia"]):
    ax3.text(v + 0.3, i, f"{v}%", va="center", fontsize=8)
ax3.set_title("% que Conclui no Último Dia\n(efeito de urgência)", fontweight="bold")
ax3.set_xlabel("% das conclusões")
ax3.axvline(20, color="gray", linestyle="--", linewidth=1, alpha=0.7)

# 4. Distribuicao de dias ate view por tipo de oferta
ax4 = fig.add_subplot(gs[1, 0])
for tipo, cor in zip(["bogo", "discount"], ["#2A6CF0", "#1D3557"]):
    subset = dias_view_pd[dias_view_pd["offer_type"] == tipo]["dias_ate_view"]
    ax4.hist(subset, bins=15, alpha=0.6, color=cor,
             label=tipo.upper(), density=True, edgecolor="white")
ax4.set_title("Distribuição — Dias até Visualização\npor Tipo de Oferta", fontweight="bold")
ax4.set_xlabel("Dias após recebimento")
ax4.set_ylabel("Densidade")
ax4.legend(fontsize=9)

# 5. Distribuicao de dias ate complete por duracao
ax5 = fig.add_subplot(gs[1, 1])
cores_dur = {5.0: "#2A6CF0", 7.0: "#1D3557", 10.0: "#A8DADC"}
for dur in sorted(dias_complete_pd["duration"].unique()):
    subset = dias_complete_pd[dias_complete_pd["duration"] == dur]["dias_ate_complete"]
    ax5.hist(subset, bins=12, alpha=0.6,
             color=cores_dur.get(dur, "#ccc"),
             label=f"{int(dur)} dias", density=True, edgecolor="white")
ax5.set_title("Distribuição — Dias até Conclusão\npor Duração da Oferta", fontweight="bold")
ax5.set_xlabel("Dias após recebimento")
ax5.set_ylabel("Densidade")
ax5.legend(fontsize=9)

plt.savefig("/Workspace/Users/matheusmartinez2018@gmail.com/coupon-optimization-ml/data/processed/eda_tempo.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Figura 3 salva.")

### Tempo de Resposta às Ofertas

A análise considera apenas os 44.286 episódios em que o cliente tinha
histórico de transações antes do recebimento da oferta. Clientes sem
compras anteriores ao envio foram excluídos por não ter base para
calcular ticket médio, recência e frequência.

Quem vai visualizar uma oferta faz isso no mesmo dia ou no dia seguinte ao
recebimento. Prazo mais longo não muda esse comportamento.

A conclusão segue padrão diferente: ofertas de 5 dias são concluídas em média
em 1,4 dias, ofertas de 10 dias chegam a 3,9 dias. O cliente não espera o prazo
expirar. O percentual de conclusão no último dia ficou abaixo de 2% em todas as
ofertas, descartando efeito de urgência como fator relevante.

A janela de resposta relevante é curta. Para o modelo, a janela completa de
validade é usada como período de observação, e episódios cuja janela não foi
integralmente observada são excluídos na etapa de modelagem.